# EEG-CNN Approximate-MAC Research — Phase A

This is the **visible, step-by-step notebook** opened in VS Code. It drives the project modules in `../src/`, shows data acquisition progress, then audits preprocessing, folds, baseline metrics, and paper figures. The prior Kaggle notebook contains the completed subject-1 smoke pilot; this local notebook is the persistent reproducible workflow.

## Acceptance checkpoints
1. Acquire 7 EDF runs for each of 105 eligible subjects (1–109 excluding 88, 89, 92, 100).
2. Verify 64 EEG channels at 160 Hz.
3. Filter 0.5–40 Hz; make 480-sample/3-second epochs and four labels (left, right, eyes-open rest, feet).
4. Save five subject-level folds and train-statistics-normalized tensors.
5. Run global five-fold evaluation before comparing to Wang et al.'s 65.07% reference.

Full fetch/preprocessing takes substantial time. Start with `SUBJECTS=[1,2,3,4,5]`, inspect the logged progress, then set `SUBJECTS=None` for the full cohort.

## 0. Runtime and project-path preflight

**What this code does:** imports only standard path/runtime helpers, reports the Python version and project root, and checks whether core packages are available. It does not fetch data, train a model, or run an experiment.

**Runtime rule:** use this notebook as the visible source in VS Code, but attach research execution to the Kaggle runtime. Do not run its research cells on the local PC. Confirm the Kaggle notebook has the project source and data paths before running the later cells; Kaggle's `/kaggle/working` paths differ from VS Code paths.


In [ ]:
from pathlib import Path
import importlib.util, sys
ROOT = Path('..').resolve()
DATA = ROOT / 'data'
print('Python:', sys.version)
print('Project:', ROOT)
for name in ['mne', 'moabb', 'torch', 'numpy', 'sklearn', 'matplotlib']:
    print(f'{name}:', bool(importlib.util.find_spec(name)))
assert all(importlib.util.find_spec(x) for x in ['mne','torch','numpy','sklearn']), 'Install ../requirements.txt before running acquisition/preprocessing.'

## 1. Dataset acquisition — full cohort

**What this code does:** fetches the seven EEGMMIDB EDF runs needed by the four-class protocol for the selected subject list, using the project's downloader and MNE's PhysioNet cache. It prints the cache location and visible download progress so interrupted acquisitions can be checked and resumed.

**How to use it:** begin with a small list such as `[1, 2, 3, 4, 5]`; use `None` only for all 105 eligible subjects. This is a network- and disk-intensive step. Run it in the Kaggle runtime, never on the local PC. The current recorded cohort status is incomplete (last observed: 530/735 files, 0 failed files); rerun/resume and verify all expected files before preprocessing.


In [ ]:
from src.data.download import download_eegmmidb
# Start with five subjects to validate the full seven-run pipeline; set None for all 105.
SUBJECTS = [1, 2, 3, 4, 5]
raw_dir = download_eegmmidb(DATA / 'raw', SUBJECTS)
print('Raw files are cached under:', raw_dir)

## 2. Verify acquired files

**What this code does:** builds the expected subject/run file list, reports missing files, and opens one representative EDF header without loading the full recording. The assertions stop the pipeline if files are absent or the expected EEG montage and 160 Hz sampling rate are not present.

**Expected result:** every selected subject has runs 01, 04, 06, 08, 10, 12, and 14; the probe has 64 channels at 160 Hz. This is an acquisition and metadata check, not a model-quality result.


In [ ]:
from src.data.download import EEGMMIDB_RUNS
import mne
from pathlib import Path
selected = SUBJECTS if SUBJECTS is not None else __import__('src.data.download', fromlist=['eligible_subjects']).eligible_subjects()
edf_root = DATA/'raw'/'MNE-eegbci-data'/'files'/'eegmmidb'/'1.0.0'
expected = [edf_root/f'S{s:03d}'/f'S{s:03d}R{r:02d}.edf' for s in selected for r in EEGMMIDB_RUNS]
missing = [p for p in expected if not p.exists()]
print(f'Expected EDFs: {len(expected)}; present: {len(expected)-len(missing)}; missing: {len(missing)}')
assert not missing, f'Missing first files: {missing[:5]}'
probe = mne.io.read_raw_edf(edf_root/'S001'/'S001R04.edf', preload=False, verbose='ERROR')
print('Run 04:', len(probe.ch_names), 'channels,', probe.info['sfreq'], 'Hz,', round(probe.n_times/probe.info['sfreq'],1), 's')
assert len(probe.ch_names)==64 and probe.info['sfreq']==160

## 3. Preprocess, map run-specific labels, and cache

**What this code does:** applies the 0.5–40 Hz bandpass, creates fixed 480-sample/3-second windows, maps imagery annotations into left/right/feet labels, and derives the rest class from non-overlapping windows of the eyes-open baseline run. It then creates deterministic subject-level folds and uses each fold's training data only to estimate channel normalization statistics before transforming both splits.

**Expected result:** finite float32 EEG tensors with shape `(trials, 64, 1, 480)`, four labels, fold manifests, and cached tensors. Subject boundaries must stay intact between train and validation to measure unseen-subject generalization.


In [ ]:
from src.data.preprocess import preprocess_eegmmidb
# Uses the same selected cohort as acquisition. A full run is 105 subjects.
cache_dir = preprocess_eegmmidb(DATA, DATA/'processed'/'eegmmidb', seed=42, subjects=SUBJECTS)
print('Fold tensor caches:', cache_dir)

## 4. Verify fold manifests and cached tensors

**What this code does:** checks train/validation subject disjointness for each saved fold, loads the cached tensors, and validates tensor shape, dtype, finite values, label coverage, and trial/label count agreement. It prints per-fold class counts to expose missing or imbalanced labels.

**Expected result:** for the full 105-subject cohort, each fold contains 84 training and 21 validation subjects. A smoke subset has smaller proportional folds. Any failed assertion means model training should not proceed.


In [ ]:
import json, torch
split_dir=DATA/'splits'
for fold in range(5):
    manifest=json.loads((split_dir/f'eegmmidb_{fold}.json').read_text())
    a,b=set(manifest['train_subjects']),set(manifest['validation_subjects'])
    assert not a & b
    pack=torch.load(DATA/'processed'/'eegmmidb'/f'eegmmidb_fold{fold}.pt',map_location='cpu',weights_only=True)
    for split in ('train','validation'):
        x,y=pack[f'{split}_x'],pack[f'{split}_y4']
        assert x.shape[1:]==(64,1,480) and x.dtype==torch.float32 and torch.isfinite(x).all()
        assert len(x)==len(y) and set(y.tolist())=={0,1,2,3}
        print(f'fold {fold} {split}: subjects={len(a) if split=="train" else len(b)}, X={tuple(x.shape)}, counts={torch.bincount(y,minlength=4).tolist()}')
print('Fold manifest/cache checks passed.')

## 5. Global EEGNet five-fold baseline

**What this code does:** marks the acceptance gate and intended command for global subject-level EEGNet training. The repository trainer is currently a stub, so this cell deliberately does not claim or fabricate a cross-validation score.

**Expected result after the trainer is implemented:** fold-level and out-of-fold metrics (accuracy, macro-F1, sensitivity/TPR, and confusion matrix) with fixed seeds and split manifests. Only compare against the 65.07% reference after confirming the class/window/split protocol is sufficiently matched. The earlier one-subject pilot is not a baseline reproduction.


In [ ]:
# Current trainer is still a stub; do not report a global baseline until implemented.
# Intended reproducible entry point:
# !python -m src.train.train_baseline --dataset eegmmidb --fold all --seed 42
print('Acceptance gate: global 5-fold training must complete before Phase B.')

## 6. Diagnostic figures for the paper

**What this code does:** writes a clearly labeled context plot comparing four-class chance, the one-subject smoke pilot, and the published global-CV reference. The bars are deliberately labeled as different protocols; they are context only and cannot establish replication or superiority.

**Expected result:** a 300 dpi PNG under `results/figures/`. Replace or supplement this pilot context after global cross-validation with out-of-fold fold variability, a normalized confusion matrix, class metrics, and subject-level robustness plots. Execute figure-generation cells on Kaggle so they use the remote runtime's data and dependencies.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
FIG_DIR=ROOT/'results'/'figures'; FIG_DIR.mkdir(parents=True,exist_ok=True)
pilot_acc=0.50
fig,ax=plt.subplots(figsize=(7,4))
labels=['Chance (4-class)','Subject 1 pilot\n(18 trials)','Wang et al.\n(global 5-fold)']
values=[25,100*pilot_acc,65.07]
bars=ax.bar(labels,values,color=['#999999','#2878B5','#54A24B'])
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,100); ax.set_title('Smoke-test context — protocols differ')
ax.bar_label(bars,fmt='%.1f%%'); ax.text(.5,-.26,'Pilot is one subject and n=18; the published result is global subject-level CV.',transform=ax.transAxes,ha='center',fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR/'pilot_vs_reference_context.png',dpi=300,bbox_inches='tight'); plt.show()
print('Saved:',FIG_DIR/'pilot_vs_reference_context.png')

## Status and acceptance criteria

- **Verified pilot only:** Kaggle subject 1; four EDFs fetched for the smoke test; 64 EEG channels at 160 Hz; 89 windows (class counts 23/22/20/24); EEGNet parameter count 3,092; held-out ACC 50.0% and macro-F1 0.462 on 18 windows. These are pilot results, not global CV.
- **Full EEGMMIDB cohort:** the last visible Kaggle acquisition log reported 530/735 EDF files and 0 failures, still running. Completion and preprocessing have not been verified.
- **BCI Competition IV 2a:** not fetched or trained in this task.
- **EvoApprox8b:** not loaded or swept; it is an approximate-circuit library, not a dataset.
- **Next gates:** complete/verify full EEGMMIDB acquisition, preprocessing and fold caches; implement and execute global five-fold baseline; then extend to BCI IV 2a and approximate arithmetic. Run research code in Kaggle only; use VS Code to inspect/edit the notebook.


## 7. Dataset and deployment comparison (metadata comparison, not a performance benchmark)

**What the following code does:** records the source-backed metadata and current verification status for EEGMMIDB and BCI IV 2a, while explicitly treating EvoApprox8b as an arithmetic-circuit library. It prints a comparison table, saves a CSV, and plots cohort size, EEG channel count, and sampling rate at 300 dpi.

**How to interpret it:** this figure compares acquisition and model-input scale only. It does not compare accuracy, runtime, energy, or deployment readiness. The datasets have different class definitions and validation protocols, so a fair model comparison requires harmonized preprocessing and evaluation. EEGMMIDB offers a larger subject cohort for cross-subject training; BCI IV 2a offers a standardized four-class imagery task and a lower EEG channel count, but only nine subjects. Neither property by itself predicts deployment performance. The EvoApprox8b software sweep can estimate model-output sensitivity; hardware-specific energy/latency claims require synthesis or measurement on the target device.

Sources: [PhysioNet EEGMMIDB](https://physionet.org/content/eegmmidb/1.0.0/), [BCI Competition IV dataset description](https://www.bbci.de/competition/iv/), [EvoApprox8b reference](https://www.fit.vut.cz/research/result/c134714/.en).


In [ ]:
# Comparison figure/table uses dataset metadata only. It is not an accuracy or deployment benchmark.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame([
    {"resource":"EEGMMIDB", "kind":"EEG dataset", "subjects":105, "eeg_channels":64, "sampling_hz":160, "window_s":3.0,
     "classes":"4 protocol classes (left/right/rest/feet)", "work_status":"subject-1 smoke test done; cohort fetch last verified 530/735"},
    {"resource":"BCI IV 2a", "kind":"EEG dataset", "subjects":9, "eeg_channels":22, "sampling_hz":250, "window_s":4.5,
     "classes":"4 MI classes (left/right/feet/tongue)", "work_status":"not fetched or trained"},
    {"resource":"EvoApprox8b", "kind":"multiplier library", "subjects":None, "eeg_channels":None, "sampling_hz":None, "window_s":None,
     "classes":"not applicable", "work_status":"not loaded or swept"},
])
print(comparison.to_string(index=False))

# Only compare like-for-like EEG acquisition/model-input dimensions.
eeg = comparison[comparison["kind"] == "EEG dataset"].copy()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
colors = ["#2878B5", "#D9534F"]
for ax, col, label in zip(axes, ["subjects", "eeg_channels", "sampling_hz"],
                          ["Available subjects (cohort scale)", "EEG channels", "Sampling rate (Hz)"]):
    ax.bar(eeg["resource"], eeg[col], color=colors)
    ax.set_title(label)
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=18)
    for i, value in enumerate(eeg[col]):
        ax.text(i, value, f"{value:g}", ha="center", va="bottom")
fig.suptitle("EEG dataset scale and input characteristics — metadata, not model results", fontsize=13)
fig.text(.5, -.015, "EEGMMIDB window: 3.0 s; BCI IV 2a window: 4.5 s. Different protocols; do not compare accuracy without harmonization.", ha="center", fontsize=9)
fig_dir = Path("/kaggle/working/eeg-axmac/results/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / "dataset_comparison_metadata.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
comparison.to_csv(fig_dir.parent / "dataset_comparison_metadata.csv", index=False)
print("Saved 300 dpi metadata comparison:", fig_path)
plt.show()
